# Clase 05 — Transformación de Datos (Transform)

**Asignatura:** Ingeniería de Datos  
**Docente:** Ing. Sergio Orozco

La **transformación** es la etapa más compleja y creativa del pipeline ETL.  
Su objetivo es convertir datos crudos en datos **limpios, consistentes, enriquecidos y listos para el análisis**.

| Sección | Tema |
|---------|------|
| **1** | Instalación e importación de librerías |
| **2** | Carga y diagnóstico del dataset crudo |
| **3** | Limpieza de datos |
| **4** | Normalización y estandarización |
| **5** | Enriquecimiento mediante JOIN |
| **6** | Derivación de columnas calculadas |
| **7** | Agregaciones y resumen |
| **8** | Pivot Table |
| **9** | Guardado de resultados |

**Tecnologías:** `pandas`, `numpy`

## 1. Instalación e Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Rutas de trabajo ──────────────────────────────────────────────────────────
DIR_INPUT  = Path("datos/input")
DIR_OUTPUT = Path("datos/output")
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Librerías importadas correctamente.")
print(f"pandas  versión: {pd.__version__}")
print(f"numpy   versión: {np.__version__}")

## 2. Carga y Diagnóstico del Dataset Crudo

Antes de transformar, debemos **entender qué tenemos**:  
tipos de datos, nulos, duplicados y valores inconsistentes.

> Los archivos de entrada se encuentran en `datos/input/`  
> y simulan datos tal como llegarían desde un sistema OLTP real.

In [ ]:
# Cargar el dataset de ventas crudas
df_crudo = pd.read_csv(DIR_INPUT / "ventas_crudas.csv")

print("=== PRIMERAS FILAS ===")
df_crudo

In [ ]:
# ── Diagnóstico inicial ────────────────────────────────────────────────────────
print("=== TIPOS DE DATO ===")
print(df_crudo.dtypes)

print(f"\n=== DIMENSIONES ===")
print(f"Filas: {len(df_crudo)}  |  Columnas: {len(df_crudo.columns)}")

print(f"\n=== VALORES NULOS POR COLUMNA ===")
print(df_crudo.isnull().sum())

print(f"\n=== FILAS COMPLETAMENTE DUPLICADAS ===")
print(f"Duplicados: {df_crudo.duplicated().sum()}")

print(f"\n=== VALORES ÚNICOS EN 'moneda' ===")
print(df_crudo["moneda"].value_counts())

### Problemas detectados en el dataset

| # | Problema | Columna | Acción |
|---|----------|---------|--------|
| 1 | Filas duplicadas | Todas | `drop_duplicates()` |
| 2 | `id_venta` nulo | `id_venta` | `dropna(subset=...)` |
| 3 | Fechas con formatos distintos y fecha inválida | `fecha_venta` | `pd.to_datetime(..., errors='coerce')` |
| 4 | Nombres con espacios y mayúsculas inconsistentes | `cliente` | `.str.strip().str.title()` |
| 5 | `moneda` con variantes: `ars`, `PESOS`, `usd` | `moneda` | `.str.upper()` + `replace()` |
| 6 | Montos negativos | `monto` | Filtrar con condición |
| 7 | `cliente` nulo | `cliente` | Imputar con `"Desconocido"` |

## 3. Limpieza de Datos

Corregimos uno a uno todos los problemas detectados.  
Siempre trabajamos sobre una **copia** del dataset original para no perder los datos crudos.

In [ ]:
# Trabajamos sobre una copia para preservar los datos originales
df = df_crudo.copy()
n_original = len(df)

# ── Paso 1: Eliminar filas completamente duplicadas ───────────────────────────
df = df.drop_duplicates()
print(f"[1] Duplicados eliminados   : {n_original - len(df)} fila(s)")

# ── Paso 2: Eliminar filas sin ID de venta (no identificables) ────────────────
antes = len(df)
df = df.dropna(subset=["id_venta"])
print(f"[2] Filas sin id_venta      : {antes - len(df)} fila(s) eliminadas")

#Eliminar id_producto nulos
antes = len(df)
df = df.dropna(subset=["id_producto"])
print(f"[3] Filas sin id_producto   : {antes - len(df)} fila(s) eliminadas")

# ── Paso 3: Convertir id_venta a entero ───────────────────────────────────────
df["id_venta"] = df["id_venta"].astype(int)

# ── Paso 4: Imputar cliente nulo con 'Desconocido' ────────────────────────────
nulos_cliente = df["cliente"].isna().sum()
df["cliente"] = df["cliente"].fillna("Desconocido")
print(f"[4] Clientes nulos imputados: {nulos_cliente} fila(s)")

# ── Paso 5: Normalizar nombre de cliente (strip + Title Case) ─────────────────
df["cliente"] = df["cliente"].str.strip().str.title()

# ── Paso 6: Normalizar moneda (mayúsculas + mapeo de variantes) ───────────────
mapa_monedas = {"PESOS": "ARS", "pesos": "ARS", "usd": "USD", "DOLARES": "USD", "dolares": "USD"}
df["moneda"] = df["moneda"].str.strip().str.upper().replace(mapa_monedas)
print(f"[6] Monedas normalizadas    : {df['moneda'].unique()}")

# ── Paso 7: Parsear fechas con manejo de errores ──────────────────────────────
#remplazo en la fecha los "-" por "/" para que el formato sea consistente y se pueda parsear correctamente
df["fecha_venta"] = df["fecha_venta"].str.replace("-", "/")
# errors='coerce' convierte los valores inválidos en NaT (Not a Time)
df["fecha_venta"] = pd.to_datetime(df["fecha_venta"], dayfirst=True, errors="coerce")
fechas_invalidas = df["fecha_venta"].isna().sum()
df = df.dropna(subset=["fecha_venta"])
print(f"[7] Fechas inválidas elim.  : {fechas_invalidas} fila(s)")

# ── Paso 8: Eliminar montos negativos ─────────────────────────────────────────
antes = len(df)
df = df[df["monto"] >= 0]
print(f"[8] Montos negativos elim.  : {antes - len(df)} fila(s)")

print(f"\n=== RESULTADO ===")
print(f"Filas originales: {n_original}  →  Filas limpias: {len(df)}")
df

## 4. Normalización y Estandarización

Normalizar significa **dar el mismo formato a valores equivalentes**.  
Es crítico cuando los datos provienen de múltiples fuentes con distintas convenciones.

En este caso, ya normalizamos `moneda` y `cliente` en el paso anterior.  
Ahora verificamos el resultado y mostramos el estado limpio del dataset.

In [ ]:
# Verificación del estado de cada columna tras la limpieza
print("=== TIPOS DE DATO DESPUÉS DE LA LIMPIEZA ===")
print(df.dtypes)

print("\n=== VALORES ÚNICOS EN 'moneda' (normalizado) ===")
print(df["moneda"].value_counts())

print("\n=== RANGO DE FECHAS ===")
print(f"Desde : {df['fecha_venta'].min().date()}")
print(f"Hasta : {df['fecha_venta'].max().date()}")

print("\n=== ESTADÍSTICAS DE MONTO ===")
print(df["monto"].describe().round(2))

## 5. Enriquecimiento mediante JOIN

El dataset de ventas tiene el `id_producto`, pero no el nombre ni la categoría.  
Cargamos el catálogo de productos y hacemos un **LEFT JOIN** para enriquecer las ventas.

> **LEFT JOIN**: mantiene todas las filas de la tabla izquierda (ventas)  
> y agrega los datos de la derecha (productos) donde el `id_producto` coincida.

In [ ]:
# Cargar el catálogo de productos
df_productos = pd.read_csv(DIR_INPUT / "catalogo_productos.csv")

print("=== CATÁLOGO DE PRODUCTOS ===")
df_productos

In [ ]:
# LEFT JOIN: enriquecer ventas con datos del catálogo
df = df.merge(df_productos, on="id_producto", how="left")

# Verificar que no quedaron productos sin match
sin_match = df["nombre"].isna().sum()
print(f"Ventas sin producto en catálogo: {sin_match}")

print("\n=== VENTAS ENRIQUECIDAS ===")
df[["id_venta", "cliente", "fecha_venta", "id_producto", "nombre", "categoria", "monto"]]

## 6. Derivación de Columnas Calculadas

Creamos columnas nuevas a partir de las existentes:  
totales, fechas derivadas y segmentación por rango de monto.

> **Regla:** nunca modifiques los valores originales. Crea columnas nuevas.

In [ ]:
# ── Columnas financieras ───────────────────────────────────────────────────────
df["total_bruto"]     = df["monto"] * df["cantidad"]
df["descuento_monto"] = df["total_bruto"] * df["descuento_pct"]
df["total_neto"]      = df["total_bruto"] - df["descuento_monto"]

# ── Columnas temporales ────────────────────────────────────────────────────────
df["anio"]       = df["fecha_venta"].dt.year
df["mes"]        = df["fecha_venta"].dt.month
df["dia_semana"] = df["fecha_venta"].dt.day_name()
df["trimestre"]  = df["fecha_venta"].dt.quarter
df["anio_mes"]   = df["fecha_venta"].dt.to_period("M")   # Período año-mes

# ── Segmentación por monto neto ───────────────────────────────────────────────
# np.select permite asignar categorías según múltiples condiciones
condiciones = [
    df["total_neto"] < 1000,
    (df["total_neto"] >= 1000) & (df["total_neto"] < 4000),
    df["total_neto"] >= 4000,
]
categorias = ["Pequeña", "Mediana", "Grande"]
df["segmento_venta"] = np.select(condiciones, categorias, default="Sin clasificar")

print("=== COLUMNAS DERIVADAS ===")
df[["id_venta", "total_bruto", "descuento_monto", "total_neto",
    "dia_semana", "trimestre", "segmento_venta"]]

## 7. Agregaciones y Resumen

Las **agregaciones** consolidan múltiples registros en métricas resumidas.  
Son la base de los reportes y dashboards de negocio.

Calculamos ventas totales, ticket promedio y cantidad de transacciones  
agrupadas por **categoría de producto**.

In [ ]:
# Agrupar por categoría y calcular métricas de negocio
resumen_categoria = (
    df.groupby("categoria")
    .agg(
        total_vendido    = ("total_neto",  "sum"),      # Suma de totales netos
        cant_transacc    = ("id_venta",    "count"),    # Cantidad de ventas
        ticket_promedio  = ("total_neto",  "mean"),     # Monto promedio por venta
        unidades_totales = ("cantidad",    "sum"),      # Unidades vendidas
    )
    .round(2)
    .sort_values("total_vendido", ascending=False)
    .reset_index()
)

print("=== VENTAS POR CATEGORÍA ===")
resumen_categoria

In [ ]:
# Agrupar por moneda para ver el total en cada divisa
resumen_moneda = (
    df.groupby("moneda")
    .agg(
        total_vendido = ("total_neto", "sum"),
        cant_ventas   = ("id_venta",   "count"),
    )
    .round(2)
    .reset_index()
)

print("=== VENTAS POR MONEDA ===")
resumen_moneda

## 8. Pivot Table

Una **tabla pivot** reorganiza los datos para comparar categorías y períodos  
en un formato matricial, muy usado en reportes de gerencia.

Construimos una tabla con **categorías en columnas** y **segmento de venta en filas**.

In [ ]:
# Pivot: filas = segmento de venta, columnas = categoría de producto
tabla_pivot = df.pivot_table(
    index   = "segmento_venta",       # Filas
    columns = "categoria",            # Columnas
    values  = "total_neto",           # Valores a agregar
    aggfunc = "sum",                  # Función de agregación
    fill_value = 0,                   # Rellenar celdas vacías con 0
).round(2)

print("=== TOTAL NETO POR SEGMENTO DE VENTA Y CATEGORÍA ===")
tabla_pivot

## 9. Guardado de Resultados

Guardamos tres archivos en `datos/output/`:

| Archivo | Contenido |
|---------|----------|
| `ventas_transformadas.csv` | Dataset completo limpio y enriquecido |
| `resumen_por_categoria.csv` | Métricas agregadas por categoría |
| `pivot_segmento_categoria.csv` | Tabla pivot para reportes |

In [ ]:
# ── 1. Dataset completo transformado ──────────────────────────────────────────
ruta_ventas = DIR_OUTPUT / "ventas_transformadas.csv"
df.to_csv(ruta_ventas, index=False, encoding="utf-8")
df.to_parquet(DIR_OUTPUT / "ventas_transformadas.parquet", index=False)
print(f"Guardado: {ruta_ventas}  ({len(df)} filas)")

# ── 2. Resumen por categoría ───────────────────────────────────────────────────
ruta_resumen = DIR_OUTPUT / "resumen_por_categoria.csv"
resumen_categoria.to_csv(ruta_resumen, index=False, encoding="utf-8")
print(f"Guardado: {ruta_resumen}  ({len(resumen_categoria)} filas)")

# ── 3. Tabla pivot ────────────────────────────────────────────────────────────
ruta_pivot = DIR_OUTPUT / "pivot_segmento_categoria.csv"
tabla_pivot.to_csv(ruta_pivot, encoding="utf-8")
print(f"Guardado: {ruta_pivot}")

print("\n✓ Todos los archivos guardados exitosamente en datos/output/")

## Resumen de la Clase

| Técnica | Función pandas / numpy | Para qué sirve |
|---------|------------------------|----------------|
| Eliminar duplicados | `drop_duplicates()` | Evitar doble conteo |
| Eliminar nulos clave | `dropna(subset=[...])` | Filas no identificables |
| Imputar nulos | `fillna(valor)` | Preservar filas con datos parciales |
| Normalizar texto | `.str.strip().str.title()` | Consistencia de nombres |
| Parsear fechas | `pd.to_datetime(..., errors='coerce')` | Unificar formatos de fecha |
| Filtrar valores | `df[condición]` | Eliminar registros inválidos |
| Reemplazar valores | `.replace(diccionario)` | Mapear variantes a valor estándar |
| JOIN | `df.merge(otro, on=col, how='left')` | Enriquecer con tablas de referencia |
| Columnas calculadas | Operaciones aritméticas sobre columnas | Totales, descuentos, márgenes |
| Fechas derivadas | `.dt.year`, `.dt.month`, `.dt.quarter` | Análisis temporal |
| Segmentación | `np.select(condiciones, categorias)` | Clasificar registros por rangos |
| Agregaciones | `.groupby().agg()` | Métricas de negocio |
| Tabla pivot | `df.pivot_table()` | Reportes matriciales |

> **Regla de oro:** Siempre trabajar sobre una copia (`df.copy()`)  
> y guardar el dataset crudo sin modificaciones para poder reprocessarlo si es necesario.